# Processing an FDN in Python

End-to-end example: load a dry signal, run it through a feedback delay network (FDN), and write a stereo reverb output to disk.

**What it does:**
- Loads a dry audio example from the package and normalizes it to float in [-1, 1].
- Configures an FDN with random delay lengths and a velvet feedback matrix (with random shifts).
- Designs one-pole absorption filters from target T60 at DC and Nyquist.
- Runs `process_fdn` and saves the result as a WAV file (and optionally plays it on macOS).

*(Python translation of `example_processFDN.m`.)*

In [ ]:
from pathlib import Path
from importlib.resources import files

import numpy as np
import soundfile as sf

import pyFDN

## Load and prepare the input signal

Read the packaged dry audio example, convert to float in [-1, 1], and append silence so the reverb tail can decay.

In [ ]:
audio_resource = files("pyFDN.audio") / "synth_dry.wav"
with audio_resource.open("rb") as f:
    x, fs = sf.read(f, dtype="float64")

x = x[:, 0] if x.ndim > 1 else x
x = np.concatenate([x, np.zeros(2 * fs, dtype=np.float64)])
x = x[0:100000]


print(f"Sampling rate: {fs} Hz")
print(f"Input length:  {x.shape[0]} samples")

## Configure the FDN structure

Set number of delay lines, random input/output gains, and build a velvet feedback matrix with random shifts. Matrix-delay approximation is computed for diagnostics.

The velvet matrix introduces a delay, so we need to add it to the delay lines for accurately computing the absorption filters


In [ ]:
rng = np.random.default_rng(seed=0)

n = 8
num_input = 1
num_output = 2

input_gain = np.ones((n, num_input))
output_gain = rng.random((num_output, n))
direct = np.zeros((num_output, num_input))
delays = rng.integers(500, 2001, size=n)

n_stages = 2
sparsity = 3
max_shift = 30
feedback_matrix, rev_feedback_matrix = pyFDN.construct_velvet_feedback_matrix(
    n, n_stages, sparsity
)
feedback_matrix, rev_feedback_matrix, _, _ = pyFDN.random_matrix_shift(
    max_shift, feedback_matrix, rev_feedback_matrix
)

matrix_delay, approx_error = pyFDN.matrix_delay_approximation(feedback_matrix)
print("Delay lengths (samples):", delays)
print("Matrix delay (samples):", np.round(matrix_delay))
print("Approximation mean abs error:", np.mean(np.abs(approx_error)))

## Design one-pole absorption filters

Use `one_pole_absorption` to get first-order IIR filters with given T60 at DC and Nyquist per delay line, then wrap them in a diagonal `ZTF` filter matrix.

In [ ]:
rt_dc = 2.0   # T60 at DC (s)
rt_ny = 0.5   # T60 at Nyquist (s)

total_delay = delays + matrix_delay
sos = pyFDN.one_pole_absorption(rt_dc, rt_ny, total_delay, fs=float(fs))
n_delays = delays.shape[0]

b_abs = np.zeros((n_delays, 1, 2), dtype=np.float64)
a_abs = np.zeros((n_delays, 1, 2), dtype=np.float64)
b_abs[:, 0, 0] = sos[0, :]
a_abs[:, 0, 0] = 1.0
a_abs[:, 0, 1] = sos[4, :]

z_absorption = pyFDN.ZTF(b_abs, a_abs, is_diagonal=True)

## Process the signal through the FDN

Run the block-based FDN processor, normalize the output to avoid clipping, save as WAV, and optionally play with `afplay` on macOS.

In [ ]:
output = pyFDN.process_fdn(
    x[:, np.newaxis],
    delays,
    feedback_matrix,
    input_gain,
    output_gain,
    direct,
    absorption_filters=z_absorption,
)
output = pyFDN.peak_normalize(output, target_peak=0.99)

out_path = Path.cwd() / "example_process_fdn_output.wav"
sf.write(out_path, output, fs)

print("Output shape:", output.shape)
print("Output RMS:", np.sqrt(np.mean(output**2)))
print("Wrote:", out_path)

from IPython.display import Audio, display
display(Audio(output.T, rate=fs))
